In [4]:
import pandas as pd
import sqlite3

In [5]:
#Carregando base de dados em um DataFrame
path = "/Users/Clarice/Desktop/Dados/Datasets p python e sql/"  
dataset = path +  'mental health sqlite/mental_health.sqlite'
conn = sqlite3.connect(dataset)

In [6]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

       name
0    Answer
1  Question
2    Survey


In [7]:
#Consultando tabela1
tabela1 = pd.read_sql_query( """
SELECT 
    *
FROM Answer
--LIMIT 10;
""", conn)
tabela1

,AnswerText,SurveyID,UserID,QuestionID
0,37,2014,1,1
1,44,2014,2,1
2,32,2014,3,1
3,31,2014,4,1
4,31,2014,5,1
...,...,...,...,...
236893,Other,2016,2689,117
236894,Support,2016,2690,117
236895,Back-end Developer,2016,2691,117
236896,DevOps/SysAdmin,2016,2692,117


In [8]:
#Consultando tabela2
tabela2 = pd.read_sql_query( """
SELECT 
    *
FROM Question
--LIMIT 20;
""", conn)
tabela2
##query.to_csv("questions.csv", index=False)

,questiontext,questionid
0,What is your age?,1
1,What is your gender?,2
2,What country do you live in?,3
3,"If you live in the United States, which state ...",4
4,Are you self-employed?,5
...,...,...
100,Do you think that team members/co-workers woul...,114
101,"If yes, what condition(s) have you been diagno...",115
102,"If maybe, what condition(s) do you believe you...",116
103,Which of the following best describes your wor...,117


In [9]:
#Consultando tabela3
tabela3 = pd.read_sql_query( """
SELECT 
    *
FROM Survey
LIMIT 20;
""", conn)
tabela3

,SurveyID,Description
0,2014,mental health survey for 2014
1,2016,mental health survey for 2016
2,2017,mental health survey for 2017
3,2018,mental health survey for 2018
4,2019,mental health survey for 2019


# Quantas pessoas responderam à pesquisa em cada ano? Quantas perguntas havia em cada ano?

In [11]:
question_1 = pd.read_sql_query( """
SELECT 
    SurveyID as "ano_pesquisa", COUNT (DISTINCT UserID) as "qtd_respondentes", COUNT (DISTINCT QuestionID) as "qtd_perguntas"
FROM Answer
GROUP BY 1
;
""", conn)
question_1

,ano_pesquisa,qtd_respondentes,qtd_perguntas
0,2014,1260,26
1,2016,1433,60
2,2017,756,76
3,2018,417,76
4,2019,352,76


A pesquisa atingiu seu maior número de participantes em 2016, mas, a partir de 2017, houve uma redução contínua, chegando a apenas 352 respondentes em 2019 (queda de 75% em relação a 2016). o número de perguntas mais que dobrou no período, isso sugere que a pesquisa ficou mais longa e detalhada, o que pode ter impactado negativamente o engajamento dos participantes.

# Qual é a taxa de retenção de respondentes que participaram da pesquisa em anos consecutivos?

In [13]:
question_2 = pd.read_sql_query( """
SELECT 
    UserID as "respondentes" , 
   COUNT (DISTINCT SurveyID) as "qtd_pesquisa"
FROM Answer
GROUP BY 1
;
""", conn)
question_2

,respondentes,qtd_pesquisa
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
...,...,...
4213,4214,1
4214,4215,1
4215,4216,1
4216,4217,1


Observamos que os UserIDs foram redefinidos a cada ano, logo cada respondente foi tratado como um novo usuário e o resultado será sempre 1 em quantidade de pesquisas respondidas. Isso impede a análise da retenção dos participantes ao longo dos anos.

In [14]:
question_2 = pd.read_sql_query( """
WITH users_per_year AS (
    SELECT DISTINCT SurveyID, UserID 
    FROM Answer
),
retained_users AS (
    SELECT 
        a.SurveyID AS "ano_atual",
        COUNT(DISTINCT a.UserID) AS "respondentes_retidos",
        COUNT(DISTINCT b.UserID) AS "respondentes_ano_anterior"
    FROM users_per_year a
    LEFT JOIN users_per_year b 
        ON a.UserID = b.UserID 
        AND a.SurveyID = b.SurveyID + 1
    GROUP BY a.SurveyID
)
SELECT 
    ano_atual,
    respondentes_retidos,
    respondentes_ano_anterior,
    ROUND((respondentes_retidos * 100.0 / NULLIF(respondentes_ano_anterior, 0)), 2) AS "taxa_retenção_%"
FROM retained_users
ORDER BY ano_atual;

""", conn)
question_2

,ano_atual,respondentes_retidos,respondentes_ano_anterior,taxa_retenção_%
0,2014,1260,0,None
1,2016,1433,0,None
2,2017,756,0,None
3,2018,417,0,None
4,2019,352,0,None


# Qual porcentagem de entrevistados vem de cada país em cada ano?

In [16]:
question_3 = pd.read_sql_query( """

WITH
total AS (
SELECT 
    SurveyID AS "ano_pesquisa", 
    COUNT (DISTINCT UserID) AS "total_respondentes"
FROM Answer
GROUP BY 1
),

country as (
SELECT 
    SurveyID AS "pesquisa_ano", 
    AnswerText AS "país",
    COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('3')
GROUP BY 1,2
)

SELECT country.*, ROUND((country."qtd_respondentes" * 100.0 / total."total_respondentes"), 2) AS "percentual"
FROM country
LEFT JOIN total ON country."pesquisa_ano" = total."ano_pesquisa"
;
""", conn)
question_3

,pesquisa_ano,país,qtd_respondentes,percentual
0,2014,Australia,22,1.75
1,2014,Austria,3,0.24
2,2014,"Bahamas, The",1,0.08
3,2014,Belgium,6,0.48
4,2014,Bosnia and Herzegovina,1,0.08
...,...,...,...,...
214,2019,Spain,3,0.85
215,2019,Switzerland,4,1.14
216,2019,Turkey,3,0.85
217,2019,United Kingdom,32,9.09


# Nas pesquisas de 2016 e 2019, quais países tiveram maior participação de respondentes?

In [18]:
question_4 = pd.read_sql_query( """

WITH 
country AS (
SELECT 
    SurveyID AS "ano_pesquisa", 
    AnswerText AS "país",
    COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID = '3' AND SurveyID IN ('2016', '2019')
GROUP BY 1,2
),

ranked_countries AS (
SELECT *,
    RANK() OVER (PARTITION BY "ano_pesquisa" ORDER BY "qtd_respondentes" DESC) AS ranking_max,
    RANK() OVER (PARTITION BY "ano_pesquisa" ORDER BY "qtd_respondentes" ASC) AS ranking_min
FROM country
)

SELECT ano_pesquisa, país, qtd_respondentes, 'Maior Participação' AS tipo
FROM ranked_countries
WHERE ranking_max <= 3  

UNION

SELECT ano_pesquisa, país, qtd_respondentes, 'Menor Participação' AS tipo
FROM ranked_countries
WHERE ranking_min <= 1

ORDER BY 1, 4, 3 DESC;
""", conn)
question_4
#question_4.to_csv("rkg.csv", index=False)

,ano_pesquisa,país,qtd_respondentes,tipo
0,2016,United States of America,840,Maior Participação
1,2016,United Kingdom,180,Maior Participação
2,2016,Canada,78,Maior Participação
3,2016,Algeria,1,Menor Participação
4,2016,Argentina,1,Menor Participação
5,2016,Bangladesh,1,Menor Participação
6,2016,Brunei,1,Menor Participação
7,2016,China,1,Menor Participação
8,2016,Costa Rica,1,Menor Participação
9,2016,Ecuador,1,Menor Participação


Observamos que houve uma redução significativa no número de respondentes, principalmente nos países com maior participação. O Estados Unidos, apresentou uma queda de aproximadamente 76%. O Reino Unido também apresentou uma grande redução, de 180 para 32 respondentes (-82%). Já o terceiro país com mais participação mudou: Canadá (78) foi substituído por Portugal (18) em 2019.
A forte redução no número de respondentes nos países líderes pode indicar mudanças no alcance da pesquisa, queda no engajamento ou um foco diferente na divulgação do estudo.
Já sobre as menores participações mostram que a pesquisa teve uma distribuição ampla, mas com baixa penetração em vários países ao longo dos anos. Países da Ásia e da América do Sul apresentam com pouca representatividade.

# Qual é a distribuição etária dos profissionais de tecnologia?

In [20]:
question_5 = pd.read_sql_query( """
SELECT 
    SurveyID AS "ano_pesquisa",
    CASE 
        WHEN AnswerText BETWEEN 16 AND 22 THEN "16 - 22 anos"
        WHEN AnswerText BETWEEN 23 AND 29 THEN "23 - 29 anos"
        WHEN AnswerText BETWEEN 30 AND 36 THEN "30 - 36 anos"
        WHEN AnswerText BETWEEN 37 AND 43 THEN "37 - 43 anos"
        WHEN AnswerText BETWEEN 44 AND 50 THEN "44 - 50 anos"
        WHEN AnswerText BETWEEN 51 AND 57 THEN "51 - 57 anos"
        WHEN AnswerText BETWEEN 58 AND 65 THEN "58 - 65 anos"
        WHEN AnswerText >= 65 THEN "65 anos ou mais"
        ELSE "invalid"
    END AS "faixa_de_idade",
     COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('1')
GROUP BY 1, 2
HAVING "Faixa de idade" != 'invalid'
;
""", conn)
question_5

,ano_pesquisa,faixa_de_idade,qtd_respondentes
0,2014,16 - 22 anos,59
1,2014,23 - 29 anos,457
2,2014,30 - 36 anos,440
3,2014,37 - 43 anos,218
4,2014,44 - 50 anos,54
5,2014,51 - 57 anos,19
6,2014,58 - 65 anos,6
7,2014,65 anos ou mais,2
8,2014,invalid,5
9,2016,16 - 22 anos,58


As pesquisas atingiram predominantemente pessoas em idade profissional ativa, entre as faixas "23 - 29 anos" e "30 - 36 anos", que representam a maioria dos respondentes em todos os anos.
Há registros de respostas classificadas como "invalid" em todos os anos analisados.
Isso pode indicar erros na coleta de dados, resistência em informar a idade real ou até mesmo respostas propositalmente erradas para não se identificar.

# Qual é a diferença na participação de homens e mulheres no setor de tecnologia ao longo do tempo?

In [22]:
question_6 = pd.read_sql_query( """

WITH
total as (
SELECT 
    SurveyID as "ano_pesquisa", COUNT (DISTINCT UserID) as "total_respondentes"
FROM Answer
GROUP BY 1
),

gender as (
SELECT
    SurveyID AS "pesquisa_ano",
    CASE
        WHEN AnswerText IN ('Cishet male', 'Guy (-ish) ^_^', 'Male', 'Male (or female)', 'Male (trans)', 'Male-ish', 'Male/genderqueer', 'Masculine', 'MALE', 'NB', 'Ostensibly Male', 'Male/androgynous', 'Masculino') THEN 'Masculino'
        WHEN AnswerText  IN ('Female', 'Female assigned at birth', 'Female-identified', 'Female-ish', 'Female/gender non-binary', 'Feminine',  'Woman-identified', 'She/her/they/them', 'Feminina') THEN 'Feminino'
        ELSE 'LGBTQIA+' 
    END AS categoria,
    COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('2')
GROUP BY 1, 2
),

percentual AS (
SELECT 
    gender.*, 
    ROUND((gender."qtd_respondentes" * 100.0 / total."total_respondentes"), 2) AS "percentual",
    LAG(gender."qtd_respondentes") 
    OVER (PARTITION BY categoria ORDER BY gender."pesquisa_ano") AS "qtd_ano_anterior"
FROM gender
LEFT JOIN total ON gender."pesquisa_ano" = total."ano_pesquisa"
)

SELECT *,
    ROUND((("qtd_respondentes" - "qtd_ano_anterior") * 100 / "qtd_ano_anterior") , 2) AS "variacao"
FROM percentual
ORDER BY 1, 2;
""", conn)
question_6

,pesquisa_ano,categoria,qtd_respondentes,percentual,qtd_ano_anterior,variacao
0,2014,Feminino,247,19.60,NaN,NaN
1,2014,LGBTQIA+,20,1.59,NaN,NaN
2,2014,Masculino,993,78.81,NaN,NaN
3,2016,Feminino,337,23.52,247.0,36.0
4,2016,LGBTQIA+,38,2.65,20.0,90.0
5,2016,Masculino,1058,73.83,993.0,6.0
6,2017,Feminino,157,20.77,337.0,-53.0
7,2017,LGBTQIA+,198,26.19,38.0,421.0
8,2017,Masculino,401,53.04,1058.0,-62.0
9,2018,Feminino,106,25.42,157.0,-32.0


A participação masculina caiu significativamente ao longo dos anos, 78,81% dos respondentes, para 50,28% em 2019. 
A representatividade LGBTQIA+ cresceu entre 2014 e 2017 (20 para 198, um aumento de 421%) e embora tenha diminuído em 2018, em 2019 a categoria ainda representava uma grande parte dos participantes.
Já a participação feminina oscilou, mas não apresentou um crescimento claro.

# Qual a porcentagem de profissionais de tecnologia que enfrentam problemas de saúde mental e possuem histórico familiar desses problemas?

In [24]:
question_7 = pd.read_sql_query( """

WITH
total as (
SELECT 
    SurveyID as "ano_pesquisa", COUNT (DISTINCT UserID) as "total_respondentes"
FROM Answer
GROUP BY 1
),

diagnostico as (
SELECT 
SurveyID AS "pesquisa_ano",
COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('6','33','32','34')
and AnswerText IN ('Yes')
GROUP BY 1
)

SELECT diagnostico.*, ROUND((diagnostico."qtd_respondentes" * 100.0 / total."total_respondentes"), 2) AS "percentual"
FROM diagnostico
LEFT JOIN total ON diagnostico."pesquisa_ano" = total."ano_pesquisa"
""", conn)
question_7


,pesquisa_ano,qtd_respondentes,percentual
0,2014,492,39.05
1,2016,974,67.97
2,2017,498,65.87
3,2018,293,70.26
4,2019,234,66.48


O percentual de profissionais que enfrentam problemas de saúde mental e possuem histórico familiar cresceu significativamente entre 2014 e 2016, estabilizando-se em um patamar alto (acima de 65%) nos anos seguintes. Esse crescimento pode estar relacionado a maior conscientização sobre saúde mental ou mudanças na metodologia da pesquisa mas também evidencia que grande parcela dos profissionais do setor enfrenta esses desafios e que o problema é consistente ao longo do tempo.

# Qual a porcentagem de funcionários que se sentem confortáveis em revelar problemas de saúde mental para seus empregadores ou colegas de trabalho?

In [26]:
question_8 = pd.read_sql_query( """

WITH
total as (
SELECT 
    SurveyID as "ano_pesquisa", COUNT (DISTINCT UserID) AS "total_respondentes"
FROM Answer
GROUP BY 1
),

seguridade as (
SELECT  
SurveyID AS "pesquisa_ano", AnswerText as "resposta",
COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('18','19')
--and AnswerText IN ('Yes')
GROUP BY 1
)

SELECT seguridade.*, ROUND((seguridade."qtd_respondentes" * 100.0 / total."total_respondentes"), 2) AS "percentual"
FROM seguridade
LEFT JOIN total ON seguridade."pesquisa_ano" = total."ano_pesquisa"
""", conn)
question_8


,pesquisa_ano,resposta,qtd_respondentes,percentual
0,2016,Maybe,1433,100.0
1,2017,Yes,756,100.0
2,2018,No,417,100.0
3,2019,Yes,352,100.0


# Existem diferenças no nível de conforto ao discutir saúde mental em comparação com saúde física no setor de tecnologia?

In [28]:
question_9 = pd.read_sql_query( """

SELECT 
SurveyID AS "ano_pesquisa", AnswerText AS "resposta",
COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('57')
GROUP BY 1;
""", conn)
question_9

,ano_pesquisa,resposta,qtd_respondentes
0,2017,Same level of comfort for each,756
1,2018,Physical health,417
2,2019,Physical health,352


# Qual a porcentagem de funcionários que recebem discussões formais sobre saúde mental e recursos por ano?

In [30]:
question_10 = pd.read_sql_query( """

SELECT 
SurveyID AS "ano_pesquisa", AnswerText AS "resposta",
COUNT(DISTINCT UserID) AS "qtd_respondentes"
FROM Answer
WHERE QuestionID IN ('15','16','25')
GROUP BY 1;
""", conn)
question_10

,ano_pesquisa,resposta,qtd_respondentes
0,2016,No,1433
1,2017,No,756
2,2018,Yes,417
3,2019,Yes,352


Os dados mostram uma oscilação na percepção sobre o conforto em discutir saúde mental, com um pico negativo em 2018, além de uma maior facilidade para falar sobre saúde física. Algumas perguntas foram introduzidas em pesquisas posteriores, e melhorias na categorização das respostas são necessárias para evitar distorções nos resultados, como na pergunta 8, onde todos responderam se se sentiam confortáveis em revelar problemas de saúde mental para seus empregadores ou colegas de trabalho como 'sim' em 2017 e no ano seguinte tivemos todas as respostas como 'não'. As discussões formais sobre o tema começaram apenas em 2018, dado a ultuma pergunta. Vemos assim, que para avançar, as empresas devem ampliar debates sobre saúde mental, promover treinamentos e criar ambientes mais acolhedores para reduzir o estigma.